In [1]:
import torch
from torch import optim,nn
from torch.utils.data import Dataset,DataLoader

In [32]:
class Data(Dataset):
    def __init__(self, train=True):
        self.X = torch.arange(-3.,3,0.1).view(-1,1)
        self.func = -3 * self.X + 5
        self.y = self.func + 0.1 * torch.randn(self.X.size())
        self.len = len(self.X)
        if train == True:
            self.y[50:55] = 20
    def __getitem__(self, index):
        return self.X[index] , self.y[index]

    def __len__(self):
        return self.len

In [33]:
train_set = Data()
validate_set = Data(train=False)

In [34]:
train_loader = DataLoader(train_set,batch_size=10,shuffle=True)
validate_loader = DataLoader(validate_set,batch_size=10,shuffle=False)

In [35]:
class LinearRegression(nn.Module):
    def __init__(self, in_features, out_features) -> None:
        super(LinearRegression,self).__init__()
        self.linear = nn.Linear(in_features,out_features)

    def forward(self,X):
        return self.linear(X)

In [36]:
criterion = nn.MSELoss()

In [37]:
n_epoches = 100
learning_rates = [0.0001,0.001,0.01,0.1,1]

In [38]:
models = []

In [39]:
validate_losses = []
train_losses = []

In [40]:
for i, learning_rate in enumerate(learning_rates):
    model = LinearRegression(1, 1)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

    epoch_train_losses = []
    epoch_val_losses = []

    for epoch in range(n_epoches):
        train_loss = 0
        for X_batch, y_batch in train_loader:
            y_hat = model(X_batch)
            loss = criterion(y_hat, y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        with torch.no_grad():
            val_loss = 0
            for X_batch, y_batch in validate_loader:
                y_hat = model(X_batch)
                loss = criterion(y_hat, y_batch)
                val_loss += loss.item()
            avg_val_loss = val_loss / len(validate_loader)

        epoch_train_losses.append(avg_train_loss)
        epoch_val_losses.append(avg_val_loss)

    train_losses.append(sum(epoch_train_losses) / len(epoch_train_losses))
    validate_losses.append(sum(epoch_val_losses) / len(epoch_val_losses))
    models.append(model)

In [42]:
for lr,loss_train,loss_validate in zip(learning_rates,train_losses,validate_losses):
    print(f'Learning rate : {lr}\tTrain Loss : {loss_train}\t Validate Loss : {loss_validate}')

Learning rate : 0.0001	Train Loss : 46.44417786399526	 Validate Loss : 13.564706087683637
Learning rate : 0.001	Train Loss : 33.00875952442486	 Validate Loss : 9.326495494370661
Learning rate : 0.01	Train Loss : 31.63675732493401	 Validate Loss : 9.122614578166976
Learning rate : 0.1	Train Loss : 61.180854803919786	 Validate Loss : 36.55193775811466
Learning rate : 1	Train Loss : nan	 Validate Loss : nan
